# Chấm gom cụm sự kiện trên tập vàng 14–15/09/2026

Notebook chạy trên **Kaggle** (GPU T4 x1, bật Internet) — tránh tràn RAM ở máy. Không cần upload dataset: dữ liệu và mô hình
fine-tune tự tải từ R2 qua **Kaggle Secrets** (xem ô 1). Có thể thay bằng Kaggle Dataset nếu không muốn dùng khoá.

**Câu hỏi:** mô hình embedding nào + thuật toán nào gom các bài thành **sự kiện** đúng nhất?

| Mô hình | Là gì |
|---|---|
| `tfidf` | **Baseline**: biểu diễn bài bằng từ (từ hiếm nặng ký). Không hiểu nghĩa — "MU" ≠ "Man Utd". Mọi mô hình phải vượt mốc này |
| `bkai_zs` | **Zero-shot**: `bkai-foundation-models/vietnamese-bi-encoder` nguyên trạng, chưa học gì từ dữ liệu của mình |
| `bkai_ft` | **Fine-tune v1** (CoSENT trên nhãn v4, seed 44) — mô hình của mình |
| `e5_zs`, `e5_ft` | Mô hình B (`intfloat/multilingual-e5-base`) zero-shot / fine-tune — bật khi đã có bản fine-tune |

| Thuật toán | Cách làm |
|---|---|
| `single_pass` | Duyệt bài theo thời gian; vào cụm (còn hoạt động trong 48h) có **độ giống trung bình** với các bài trong cụm cao nhất nếu ≥ ngưỡng, không thì mở cụm mới. Baseline bắt buộc, giống cách chạy thật (bài đến từng bài) |
| `agglo` | Gom lại cả cửa sổ: agglomerative **average-linkage** trên khoảng cách 1 − cosine, cặp cách nhau > 48h bị đẩy xa. Cắt cây ở ngưỡng |

**Giao thức (chống rò rỉ):** gom trên cả hai ngày (sự kiện kéo dài qua đêm); **chọn ngưỡng trên bài ngày 14/09**; **báo cáo trên
bài ngày 15/09** — B-Cubed P/R/F1, CI 95% bootstrap theo cụm vàng, so ghép cặp giữa các mô hình. Bỏ bài `x` (bài tổng hợp).
Thêm: recall theo **tầng trùng chữ** (giá trị thật của fine-tune nằm ở cụm viết khác chữ) và **độ nhạy** (tách các cụm Claude
"không chắc" ra — thứ hạng mô hình có đổi không).

⚠ Đáp án do Claude đề xuất + kiểm tra lại, **chưa có người duyệt**; mô hình fine-tune học từ nhãn Claude → có thể được lợi.

## 0. Cấu hình

Bật/tắt mô hình trong `MODELS`. `e5_ft`: điền đường dẫn thư mục mô hình (vd. output của notebook `finetune_e5_kaggle`) rồi đổi
`on=True`. Chạy thử nhanh: `SMOKE=True` (chỉ TF-IDF + lưới ngưỡng thưa).

In [ ]:
import os
CFG = dict(
    tune_day="2026-09-14",            # chọn ngưỡng
    test_day="2026-09-15",            # báo cáo
    window_hours=48,
    grid=[round(0.05 + 0.01 * k, 2) for k in range(91)],   # ngưỡng thử: 0,05 → 0,95
    n_boot=1000,
    r2_gold="gold_clusters_NguyenDinhHieu/v1/",
    r2_model="models/embedding_v1_NguyenDinhHieu/model/",
)
SMOKE = os.environ.get("SMOKE", "0") == "1"
MODELS = [
    # tên        nguồn mô hình                                   tiền xử lý  bật?
    dict(name="tfidf",   src=None,                                     prep="raw",   on=True),
    dict(name="bkai_zs", src="bkai-foundation-models/vietnamese-bi-encoder", prep="pyvi", on=True),
    dict(name="bkai_ft", src="r2:" + CFG["r2_model"],                  prep="pyvi",  on=True),
    dict(name="e5_zs",   src="intfloat/multilingual-e5-base",          prep="query", on=True),
    dict(name="e5_ft",   src="/kaggle/input/…/e5-v4-cosent-sXX",       prep="query", on=False),
]
ALGOS = ["single_pass", "agglo"]
if os.environ.get("MODELS_ON"):                              # vd. MODELS_ON=tfidf,bkai_ft khi chạy thử ngoài Kaggle
    for m in MODELS:
        m["on"] = m["name"] in os.environ["MODELS_ON"].split(",")
if SMOKE:
    MODELS = [m for m in MODELS if m["name"] == "tfidf"]
    CFG.update(grid=CFG["grid"][::10], n_boot=100)
[m["name"] for m in MODELS if m["on"]]

## 1. Môi trường + dữ liệu

**Kaggle Secrets** (Add-ons → Secrets, bật cho notebook): `R2_ACCOUNT_ID`, `R2_ACCESS_KEY_ID`, `R2_SECRET_ACCESS_KEY`, `R2_BUCKET`
(= `vietnam-news-datalake`). Nên dùng **khoá chỉ đọc**. Không dán khoá vào ô code.

Không dùng R2: tạo Kaggle Dataset gồm `articles.jsonl` + `gold_clusters_v2.csv` (từ `data/gold_clusters/v1/`) và, nếu cần,
thư mục mô hình fine-tune — notebook tự tìm trong `/kaggle/input`.

In [ ]:
os.environ.setdefault("CUDA_VISIBLE_DEVICES", "0")
os.environ["TOKENIZERS_PARALLELISM"] = "false"
import importlib.util, subprocess, sys
need = [p for p, m in [("pyvi", "pyvi"), ("boto3", "boto3")] if importlib.util.find_spec(m) is None]
if need:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *need])

import csv, json, gc, time, unicodedata
from collections import Counter, defaultdict
from datetime import datetime
from pathlib import Path
import numpy as np

KAGGLE = Path("/kaggle/input").exists()
WORK = Path("/kaggle/working") if KAGGLE else Path(os.environ.get("OUT_DIR", "out_cluster"))
DATA = WORK / "data"; DATA.mkdir(parents=True, exist_ok=True)


def secrets():
    keys = ["R2_ACCOUNT_ID", "R2_ACCESS_KEY_ID", "R2_SECRET_ACCESS_KEY", "R2_BUCKET"]
    try:
        from kaggle_secrets import UserSecretsClient
        c = UserSecretsClient()
        vals = {k: c.get_secret(k) for k in keys}
    except Exception:
        vals = {k: os.environ.get(k) for k in keys}       # chạy ngoài Kaggle: lấy từ biến môi trường
    return vals if all(vals.values()) else None


def r2_download(prefix, dest, only=None):
    import boto3
    s = secrets()
    if s is None:
        raise RuntimeError("Thiếu Kaggle Secrets R2_* — thêm secrets hoặc dùng Kaggle Dataset (xem ô trên)")
    s3 = boto3.client("s3", endpoint_url=f"https://{s['R2_ACCOUNT_ID']}.r2.cloudflarestorage.com", region_name="auto",
                      aws_access_key_id=s["R2_ACCESS_KEY_ID"], aws_secret_access_key=s["R2_SECRET_ACCESS_KEY"])
    n = 0
    for page in s3.get_paginator("list_objects_v2").paginate(Bucket=s["R2_BUCKET"], Prefix=prefix):
        for o in page.get("Contents", []):
            rel = o["Key"][len(prefix):]
            if not rel or (only and rel not in only):
                continue
            out = Path(dest) / rel
            out.parent.mkdir(parents=True, exist_ok=True)
            if not out.exists() or out.stat().st_size != o["Size"]:
                s3.download_file(s["R2_BUCKET"], o["Key"], str(out)); n += 1
    return n


def find_input(name):
    for d, _, fs in os.walk("/kaggle/input", followlinks=True) if KAGGLE else []:
        if name in fs:
            return Path(d) / name
    return None


for name in ["articles.jsonl", "gold_clusters_v2.csv"]:
    src = find_input(name) or (Path(os.environ["GOLD_DIR"]) / name if "GOLD_DIR" in os.environ else None)
    if src and src.exists():
        (DATA / name).write_bytes(src.read_bytes())
    elif not (DATA / name).exists():
        print("tải từ R2:", name, r2_download(CFG["r2_gold"], DATA, only={name}))
print("dữ liệu ở", DATA, sorted(p.name for p in DATA.iterdir()))

In [ ]:
arts = [json.loads(l) for l in open(DATA / "articles.jsonl", encoding="utf-8")]
N = len(arts)
gold_rows = list(csv.DictReader(open(DATA / "gold_clusters_v2.csv", encoding="utf-8")))
gold_of = {r["article_id"]: r["cluster_id"] for r in gold_rows}
conf_of = {r["cluster_id"]: r["confidence"] for r in gold_rows}
times = np.array([datetime.fromisoformat(a["t"]).timestamp() for a in arts])
day = np.array([a["t"][:10] for a in arts])
keep = np.array([gold_of[a["id"]] != "x" for a in arts])            # bài loại không chấm
GOLD = np.array([gold_of[a["id"]] for a in arts], dtype=object)   # object: nhãn tách "E0001#12" không bị cắt
TUNE = np.where(keep & (day == CFG["tune_day"]))[0]
TEST = np.where(keep & (day == CFG["test_day"]))[0]
sizes = Counter(GOLD[keep])
print(f"{N} bài · loại {int((~keep).sum())} · {len(sizes)} cụm vàng ({sum(v >= 2 for v in sizes.values())} cụm ≥ 2 bài)")
print(f"chọn ngưỡng trên {len(TUNE)} bài ngày {CFG['tune_day']} · báo cáo trên {len(TEST)} bài ngày {CFG['test_day']}")

## 2. Vector cho từng mô hình

Văn bản = `tiêu đề. sapo` (giống lúc fine-tune). Tiền xử lý **bắt buộc** theo mô hình: bkai → NFC + tách từ `pyvi`; e5 → NFC +
tiền tố `"query: "`; TF-IDF → văn bản thô (từ đơn + cặp hai từ). Vector chuẩn hoá L2 → cosine = tích vô hướng.
Mỗi mô hình lưu `emb_<tên>.npz` để chạy lại không phải encode.

In [ ]:
def text(a):
    return f"{a['title']}. {a['sapo']}".strip(" .") if a.get("sapo") else a["title"]


def prep(t, how):
    t = unicodedata.normalize("NFC", t)
    if how == "pyvi":
        from pyvi import ViTokenizer
        return ViTokenizer.tokenize(t)
    if how == "query":
        return "query: " + t
    return t


def model_path(src):
    if src.startswith("r2:"):
        dest = WORK / "model_r2" / src[3:].strip("/").replace("/", "_")
        for d, _, fs in os.walk("/kaggle/input", followlinks=True) if KAGGLE else []:   # ưu tiên Kaggle input tên có "bkai"
            if "model.safetensors" in fs and "config_sentence_transformers.json" in fs and "bkai" in d.lower():
                return d
        print("tải mô hình từ R2:", r2_download(src[3:], dest), "file")
        return str(dest)
    return src


def embed(m):
    cache = WORK / f"emb_{m['name']}.npz"
    if cache.exists():
        z = np.load(cache, allow_pickle=True)
        if list(z["ids"]) == [a["id"] for a in arts]:
            return z["E"]
    texts = [prep(text(a), m["prep"]) for a in arts]
    if m["src"] is None:                                     # TF-IDF
        from sklearn.feature_extraction.text import TfidfVectorizer
        X = TfidfVectorizer(ngram_range=(1, 2), sublinear_tf=True, token_pattern=r"(?u)\b\w+\b").fit_transform(texts)
        E = X.astype(np.float32)                             # thưa, đã chuẩn hoá L2
        from scipy import sparse
        sparse.save_npz(WORK / f"emb_{m['name']}_sparse.npz", E)
        return E
    import torch
    from sentence_transformers import SentenceTransformer
    model = SentenceTransformer(model_path(m["src"]), device="cuda" if torch.cuda.is_available() else "cpu")
    model.max_seq_length = 256
    E = model.encode(texts, batch_size=64, normalize_embeddings=True, show_progress_bar=False).astype(np.float32)
    np.savez(cache, ids=np.array([a["id"] for a in arts]), E=E)
    del model; gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return E


SIM = {}
for m in [m for m in MODELS if m["on"]]:
    t0 = time.time()
    E = embed(m)
    S = (E @ E.T)
    S = S.toarray() if hasattr(S, "toarray") else S
    SIM[m["name"]] = np.asarray(S, dtype=np.float32)
    off = SIM[m["name"]][np.triu_indices(N, 1)]
    print(f"{m['name']:8s} {time.time() - t0:5.1f}s · cosine trung vị {np.median(off):.3f} · p99 {np.percentile(off, 99):.3f}")

## 3. Thuật toán gom cụm

Cả hai thuật toán chỉ cần ma trận cosine `S` (N × N) và thời gian → dùng chung cho mọi mô hình.
`single_pass` cập nhật dần **tổng cột** của mỗi cụm nên mỗi lần chạy chỉ ~O(N²) phép cộng (nhanh, thử được nhiều ngưỡng).

In [ ]:
from scipy.cluster.hierarchy import fcluster, linkage
from scipy.spatial.distance import squareform

ORDER = np.argsort(times, kind="stable")
WIN = CFG["window_hours"] * 3600


def single_pass(S, th):
    lab = np.full(N, -1)
    csum = np.zeros((N, N), dtype=np.float32)      # cột c = tổng S[:, thành viên cụm c]
    size, last = [], []
    for i in ORDER:
        k = len(size)
        if k:
            score = csum[i, :k] / np.array(size)
            score[times[i] - np.array(last) > WIN] = -1        # cụm đã "nguội" quá 48h
            c = int(np.argmax(score))
            if score[c] >= th:
                lab[i] = c; csum[:, c] += S[:, i]; size[c] += 1; last[c] = max(last[c], times[i])
                continue
        lab[i] = k; csum[:, k] = S[:, i]; size.append(1); last.append(times[i])
    return lab


_Z = {}
def agglo(S, th, name):
    if name not in _Z:                                       # cây chỉ dựng một lần cho mỗi mô hình
        D = 1.0 - S
        D[np.abs(times[:, None] - times[None, :]) > WIN] = 2.0  # cặp cách nhau > 48h: rất xa
        np.fill_diagonal(D, 0)
        _Z[name] = linkage(squareform(np.clip((D + D.T) / 2, 0, None), checks=False), method="average")
    return fcluster(_Z[name], t=1.0 - th, criterion="distance")


def run(name, algo, th):
    return single_pass(SIM[name], th) if algo == "single_pass" else agglo(SIM[name], th, name)

## 4. Chỉ số

**B-Cubed:** với mỗi bài, precision = tỉ lệ bài cùng cụm dự đoán mà cũng cùng cụm vàng; recall = tỉ lệ bài cùng cụm vàng được
dự đoán cùng cụm. Tính trên đúng tập bài đang chấm (ngày 14 hoặc 15/09). **CI 95%:** bootstrap lấy mẫu lại **cụm vàng** (bài
cùng sự kiện không độc lập với nhau), giữ điểm từng bài cố định; **so ghép cặp:** cùng một mẫu cho hai mô hình → CI của hiệu số.

In [ ]:
def per_item(pred, idx, gold=GOLD):
    p, g = pred[idx], gold[idx]
    n_p, n_g, n_pg = Counter(p), Counter(g), Counter(zip(p, g))
    P = np.array([n_pg[(a, b)] / n_p[a] for a, b in zip(p, g)])
    R = np.array([n_pg[(a, b)] / n_g[b] for a, b in zip(p, g)])
    return P, R


def f1(P, R, w=None):
    p, r = np.average(P, weights=w), np.average(R, weights=w)
    return 2 * p * r / (p + r) if p + r else 0.0


def boot_weights(idx, n_boot, seed=0, gold=GOLD):
    # mỗi lần: rút có hoàn lại các cụm vàng của tập idx → trọng số mỗi bài = số lần cụm của nó được rút
    rng = np.random.default_rng(seed)
    cl = sorted(set(gold[idx])); pos = {c: k for k, c in enumerate(cl)}
    item_cl = np.array([pos[c] for c in gold[idx]])
    draws = rng.integers(0, len(cl), size=(n_boot, len(cl)))
    counts = np.stack([np.bincount(d, minlength=len(cl)) for d in draws])
    return counts[:, item_cl]                                 # n_boot × len(idx)


def score(pred, idx, W=None, gold=GOLD):
    P, R = per_item(pred, idx, gold)
    out = dict(precision=float(P.mean()), recall=float(R.mean()), f1=float(f1(P, R)), n=int(len(idx)))
    if W is not None:
        fs = np.array([f1(P, R, w) if w.sum() else np.nan for w in W])
        out["f1_ci95"] = [float(np.nanpercentile(fs, 2.5)), float(np.nanpercentile(fs, 97.5))]
        out["_boot"] = fs
    return out

## 5. Chọn ngưỡng trên ngày 14/09

In [ ]:
names = [m["name"] for m in MODELS if m["on"]]
CURVE, BEST = {}, {}
for name in names:
    for algo in ALGOS:
        t0 = time.time()
        f = [score(run(name, algo, th), TUNE)["f1"] for th in CFG["grid"]]
        CURVE[(name, algo)] = f
        k = int(np.argmax(f))
        BEST[(name, algo)] = CFG["grid"][k]
        print(f"{name:8s} {algo:11s} ngưỡng tốt nhất {CFG['grid'][k]:.2f} · F1 (14/09) {f[k]:.3f} · {time.time() - t0:4.1f}s")

In [ ]:
# F1 theo ngưỡng trên ngày chọn ngưỡng — mỗi thuật toán một khung; nhãn ở đầu mút + bảng số ở ô sau
import matplotlib.pyplot as plt
PAL = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4"]       # thứ tự cố định, đã kiểm tra mù màu
fig, axes = plt.subplots(1, len(ALGOS), figsize=(6.2 * len(ALGOS), 3.8), sharey=True)
for ax, algo in zip(np.atleast_1d(axes), ALGOS):
    for k, name in enumerate(names):
        y = CURVE[(name, algo)]
        ax.plot(CFG["grid"], y, color=PAL[k % len(PAL)], lw=2, solid_capstyle="round")
        b = BEST[(name, algo)]; yb = y[CFG["grid"].index(b)]
        ax.plot([b], [yb], "o", ms=6, color=PAL[k % len(PAL)], mec="white", mew=2)
        ax.annotate(f"{name} {yb:.2f}", (b, yb), xytext=(4, 4), textcoords="offset points", fontsize=8, color="#52514e")
    ax.set_title(algo, fontsize=11, loc="left"); ax.set_xlabel("ngưỡng cosine"); ax.grid(axis="y", color="#e1e0d9", lw=1)
    for s in ("top", "right"):
        ax.spines[s].set_visible(False)
np.atleast_1d(axes)[0].set_ylabel("B-Cubed F1 (14/09)")
from matplotlib.lines import Line2D
fig.legend([Line2D([0], [0], color=PAL[k % len(PAL)], lw=2) for k in range(len(names))], names,
           loc="upper center", ncol=len(names), frameon=False, fontsize=9)
fig.tight_layout(rect=(0, 0, 1, 0.9)); plt.show()

## 6. Kết quả trên ngày 15/09 (số báo cáo)

`ΔF1 [CI]` > 0 và CI không chứa 0 → hơn hẳn. Đọc theo thứ tự: fine-tune vs zero-shot của **cùng** mô hình (fine-tune có tác dụng
không) → so với TF-IDF (embedding có đáng không) → single-pass vs agglo.

In [ ]:
W = boot_weights(TEST, CFG["n_boot"])
RES, PRED = {}, {}
for name in names:
    for algo in ALGOS:
        PRED[(name, algo)] = pred = run(name, algo, BEST[(name, algo)])
        RES[(name, algo)] = score(pred, TEST, W)
rows = []
for (name, algo), r in sorted(RES.items(), key=lambda kv: -kv[1]["f1"]):
    rows.append(dict(model=name, algo=algo, threshold=BEST[(name, algo)], precision=round(r["precision"], 3),
                     recall=round(r["recall"], 3), f1=round(r["f1"], 3),
                     f1_ci95=f"[{r['f1_ci95'][0]:.3f}, {r['f1_ci95'][1]:.3f}]",
                     n_pred_clusters=len(set(PRED[(name, algo)][TEST]))))
import pandas as pd
trivial = score(np.arange(N), TEST, W)                        # mốc tầm thường: mỗi bài một cụm
rows.append(dict(model="(mỗi bài một cụm)", algo="-", threshold=None, precision=round(trivial["precision"], 3),
                 recall=round(trivial["recall"], 3), f1=round(trivial["f1"], 3),
                 f1_ci95=f"[{trivial['f1_ci95'][0]:.3f}, {trivial['f1_ci95'][1]:.3f}]", n_pred_clusters=len(TEST)))
print(f"Tập đo: {len(TEST)} bài ngày {CFG['test_day']}, {len(set(GOLD[TEST]))} cụm vàng — phần lớn 1 bài, nên mốc "
      f"'mỗi bài một cụm' đã đạt F1 {trivial['f1']:.3f}: so mô hình với mốc này, không so với 0")
table = pd.DataFrame(rows)
table

In [ ]:
def delta(a, b):
    ra, rb = RES[a], RES[b]
    d = rb["_boot"] - ra["_boot"]
    return dict(delta=round(rb["f1"] - ra["f1"], 3), ci95=[round(float(np.nanpercentile(d, 2.5)), 3),
                round(float(np.nanpercentile(d, 97.5)), 3)], p_le_0=round(float(np.mean(d <= 0)), 3))

PAIRS = []
for algo in ALGOS:
    for a, b in [("bkai_zs", "bkai_ft"), ("e5_zs", "e5_ft"), ("tfidf", "bkai_zs"), ("tfidf", "bkai_ft"),
                 ("tfidf", "e5_zs"), ("bkai_ft", "e5_ft")]:
        if a in names and b in names:
            PAIRS.append(dict(algo=algo, so_sanh=f"{b} − {a}", **delta((a, algo), (b, algo))))
for name in names:
    PAIRS.append(dict(algo="agglo − single_pass", so_sanh=name, **delta((name, "single_pass"), (name, "agglo"))))
pd.DataFrame(PAIRS)

## 7. Phân tích bắt buộc

**Tầng trùng chữ:** mỗi cụm vàng ≥ 2 bài (ngày 15/09) được đo độ trùng chữ = cosine TF-IDF trung bình giữa các bài của nó, chia 3
tầng bằng nhau. Recall B-Cubed ở tầng **thấp** = khả năng gom các bài cùng sự kiện **viết khác chữ** — chỗ TF-IDF thua và
fine-tune phải thắng. **Độ nhạy:** tách mọi cụm Claude ghi "không chắc" thành từng bài rồi chấm lại (không chọn lại ngưỡng).

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
Xt = TfidfVectorizer(ngram_range=(1, 2), sublinear_tf=True, token_pattern=r"(?u)\b\w+\b").fit_transform([text(a) for a in arts])
St = (Xt @ Xt.T).toarray()
test_set = set(TEST.tolist())
lex = {}
for c in set(GOLD[TEST]):
    mem = [i for i in TEST if GOLD[i] == c]
    if len(mem) >= 2:
        sub = St[np.ix_(mem, mem)]
        lex[c] = float(sub[np.triu_indices(len(mem), 1)].mean())
cut = np.quantile(list(lex.values()), [1 / 3, 2 / 3])
tier = {c: ("thấp" if v < cut[0] else "vừa" if v < cut[1] else "cao") for c, v in lex.items()}
print(f"{len(lex)} cụm ≥ 2 bài · ngưỡng tầng trùng chữ {cut.round(3)}")

GOLD_SPLIT = GOLD.copy()                                       # độ nhạy: tách cụm "không chắc"
for i in range(N):
    if conf_of.get(GOLD[i]) == "thấp":
        GOLD_SPLIT[i] = f"{GOLD[i]}#{i}"

rows = []
for (name, algo), pred in PRED.items():
    P, R = per_item(pred, TEST)
    row = dict(model=name, algo=algo)
    for t in ("thấp", "vừa", "cao"):
        mask = np.array([tier.get(GOLD[i]) == t for i in TEST])
        row[f"recall_{t}"] = round(float(R[mask].mean()), 3)
    row["f1_goc"] = round(RES[(name, algo)]["f1"], 3)
    row["f1_tach_khong_chac"] = round(score(pred, TEST, gold=GOLD_SPLIT)["f1"], 3)
    rows.append(row)
analysis = pd.DataFrame(rows).sort_values("f1_goc", ascending=False)
analysis

## 8. Lưu kết quả + danh sách lỗi để đọc (bước phân tích lỗi)

`loi_<mô hình>_<thuật toán>.csv`: các cụm dự đoán **gộp nhầm** (chứa bài của ≥ 2 sự kiện vàng) và các sự kiện vàng bị **tách
nhầm**, sắp theo số bài ảnh hưởng — đọc ~30 dòng đầu, ghi nguyên nhân (cùng câu chuyện / cùng khuôn tin / viết khác chữ / trôi tâm).

In [ ]:
best_name, best_algo = table.iloc[0]["model"], table.iloc[0]["algo"]
out = dict(config=CFG, models=[m for m in MODELS if m["on"]], thresholds={f"{k[0]}|{k[1]}": v for k, v in BEST.items()},
           test={f"{k[0]}|{k[1]}": {kk: vv for kk, vv in r.items() if kk != "_boot"} for k, r in RES.items()},
           table=table.to_dict("records"), pairs=PAIRS,
           analysis=analysis.to_dict("records"), curves={f"{k[0]}|{k[1]}": v for k, v in CURVE.items()},
           note="Đáp án: gold_clusters_v2.csv (Claude đề xuất + kiểm tra, chưa có người duyệt).")
(WORK / "results_clustering.json").write_text(json.dumps(out, ensure_ascii=False, indent=2, default=str), encoding="utf-8")
table.to_csv(WORK / "bang_ket_qua.csv", index=False)

def errors(name, algo):
    pred = PRED[(name, algo)]; rows = []
    for c in sorted(set(pred[TEST])):                                   # gộp nhầm
        mem = [i for i in TEST if pred[i] == c]; g = Counter(GOLD[mem])
        if len(g) >= 2:
            rows.append(dict(loai="gop_nham", so_bai=len(mem), so_su_kien_vang=len(g),
                             bai=" | ".join(f"[{arts[i]['stt']}·{GOLD[i]}] {arts[i]['title'][:70]}" for i in mem)))
    for c in sorted(set(GOLD[TEST])):                                   # tách nhầm
        mem = [i for i in TEST if GOLD[i] == c]; p = Counter(pred[mem])
        if len(p) >= 2:
            rows.append(dict(loai="tach_nham", so_bai=len(mem), so_cum_du_doan=len(p),
                             bai=" | ".join(f"[{arts[i]['stt']}·cụm {pred[i]}] {arts[i]['title'][:70]}" for i in mem)))
    rows.sort(key=lambda r: -r["so_bai"])
    pd.DataFrame(rows).to_csv(WORK / f"loi_{name}_{algo}.csv", index=False)
    return len(rows)

for key in [(best_name, best_algo)] + [(n, best_algo) for n in ("bkai_zs", "tfidf") if n in names and n != best_name]:
    print("lỗi", key, errors(*key), "dòng")
with open(WORK / f"du_doan_{best_name}_{best_algo}.csv", "w", encoding="utf-8", newline="") as f:
    w = csv.writer(f); w.writerow(["article_id", "stt", "published_at", "pred_cluster", "gold_cluster", "title"])
    for i in ORDER:
        w.writerow([arts[i]["id"], arts[i]["stt"], arts[i]["t"], int(PRED[(best_name, best_algo)][i]), GOLD[i], arts[i]["title"]])
print("cấu hình tốt nhất:", best_name, best_algo, "· đã ghi", sorted(p.name for p in WORK.iterdir() if p.is_file()))

## 9. Lấy kết quả về

**Save Version → Save & Run All** để chạy nền; xong vào tab *Output* tải `results_clustering.json`, `bang_ket_qua.csv`,
`loi_*.csv`, `du_doan_*.csv`, đặt vào `data/temp/kaggle_runs/<ngày>_cluster-eval/`. Các file `emb_*.npz` dùng lại được cho bước
chạy toàn kho (không cần encode lại 1.496 bài này).